# EDA — Análisis Exploratorio de Datos

## BEKANTOR Demand Intelligence

El objetivo de esta etapa es explorar el comportamiento de las ventas,
detectar patrones, relaciones, valores atípicos y posibles problemas
que deberán tratarse posteriormente durante la limpieza profunda y
la ingeniería de variables.

## Librerías y configuración

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

### Conexión del notebook con el proyecto

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT) 

c:\Users\keebs\OneDrive\Documentos\INTELIGENCIA ARTIFICIAL\MATERIAS 5S\PRACTICA PROFESIONALIZANTE 2\pf\bekantor-demand-intelligence


In [3]:
from src.data.load_data import load_raw_data
from src.data.clean_data import minimal_cleaning

datasets = load_raw_data()
datasets = minimal_cleaning(datasets)

sales = datasets["sales"]
stores = datasets["stores"]
transactions = datasets["transactions"]
holidays = datasets["holidays"]
oil = datasets["oil"]

In [4]:
print(f"Filas oil después de completar calendario: {len(oil):,}")
print(f"Nulos restantes: {oil['dcoilwtico'].isna().sum()}")
print(f"Desde: {oil['date'].min().date()}")
print(f"Hasta: {oil['date'].max().date()}")

Filas oil después de completar calendario: 1,688
Nulos restantes: 1
Desde: 2013-01-01
Hasta: 2017-08-15


In [5]:
from src.data.panel_diagnostics import (
    add_store_operation_flags,
    add_zero_taxonomy,
    find_missing_dates
)

## 6.0 Diagnóstico estructural del panel temporal

Antes de analizar la distribución de las ventas se verifica la estructura temporal del panel, las fechas de apertura de las tiendas y la naturaleza de los registros con ventas iguales a cero.

In [6]:
sales, apertura = add_store_operation_flags(sales)

apertura.sort_values()

store_nbr
25   2013-01-01
28   2013-01-02
30   2013-01-02
31   2013-01-02
32   2013-01-02
33   2013-01-02
34   2013-01-02
35   2013-01-02
37   2013-01-02
38   2013-01-02
39   2013-01-02
1    2013-01-02
41   2013-01-02
43   2013-01-02
44   2013-01-02
45   2013-01-02
46   2013-01-02
47   2013-01-02
48   2013-01-02
49   2013-01-02
50   2013-01-02
51   2013-01-02
26   2013-01-02
40   2013-01-02
27   2013-01-02
23   2013-01-02
2    2013-01-02
3    2013-01-02
4    2013-01-02
5    2013-01-02
6    2013-01-02
7    2013-01-02
8    2013-01-02
9    2013-01-02
10   2013-01-02
11   2013-01-02
24   2013-01-02
13   2013-01-02
12   2013-01-02
15   2013-01-02
16   2013-01-02
17   2013-01-02
18   2013-01-02
19   2013-01-02
14   2013-01-02
54   2013-01-02
36   2013-05-09
53   2014-05-29
20   2015-02-13
29   2015-03-20
21   2015-07-24
42   2015-08-21
22   2015-10-09
52   2017-04-20
Name: date, dtype: datetime64[us]

In [7]:
dataset_start = sales["date"].min()

late_openings = apertura[
    apertura > pd.Timestamp("2013-01-02")
].sort_values()

late_openings

store_nbr
36   2013-05-09
53   2014-05-29
20   2015-02-13
29   2015-03-20
21   2015-07-24
42   2015-08-21
22   2015-10-09
52   2017-04-20
Name: date, dtype: datetime64[us]

In [8]:
pre_opening = (
    (sales["sales"] == 0) &
    (~sales["tienda_operativa"])
)

print(f"Ceros pre-apertura: {pre_opening.sum():,}")
print(
    f"Porcentaje del dataset: "
    f"{pre_opening.mean() * 100:.2f}%"
)

Ceros pre-apertura: 222,057
Porcentaje del dataset: 7.40%


In [9]:
sales = add_zero_taxonomy(sales)

In [10]:
zero_summary = (
    sales["tipo_cero"]
    .value_counts()
    .rename_axis("tipo")
    .reset_index(name="registros")
)

zero_summary["porcentaje"] = (
    zero_summary["registros"]
    / len(sales)
    * 100
)

zero_summary

,tipo,registros,porcentaje
0,Venta positiva,2061758,68.704930
1,Cero real de demanda,635449,21.175365
2,Pre-apertura,222057,7.399710
3,Familia no surtida,81624,2.719995


In [11]:
missing_dates = find_missing_dates(sales)

print(f"Cantidad de fechas faltantes: {len(missing_dates)}")

for date in missing_dates:
    print(date.date())

Cantidad de fechas faltantes: 4
2013-12-25
2014-12-25
2015-12-25
2016-12-25


In [12]:
# Tiendas ya operativas
sales_valid = sales[
    sales["tienda_operativa"]
].copy()

# Panel realmente modelable:
# tienda operativa + familia que esa tienda efectivamente comercializa
sales_modelable = sales[
    sales["tienda_operativa"] &
    sales["familia_surtida"]
].copy()

print(f"Panel original:   {len(sales):,}")
print(f"Panel operativo:  {len(sales_valid):,}")
print(f"Panel modelable:  {len(sales_modelable):,}")

Panel original:   3,000,888
Panel operativo:  2,778,831
Panel modelable:  2,697,207


## 6.1 Distribución de las ventas

In [13]:
sales_modelable.head()

,id,date,store_nbr,family,sales,onpromotion,tienda_operativa,familia_surtida,tipo_cero
1729,581,2013-01-01,25,LAWN AND GARDEN,2.0,0,True,True,Venta positiva
1731,579,2013-01-01,25,HOME CARE,0.0,0,True,True,Cero real de demanda
1732,578,2013-01-01,25,HOME APPLIANCES,0.0,0,True,True,Cero real de demanda
1733,577,2013-01-01,25,HOME AND KITCHEN II,0.0,0,True,True,Cero real de demanda
1734,576,2013-01-01,25,HOME AND KITCHEN I,0.0,0,True,True,Cero real de demanda


In [14]:
# Resumen sobre el panel realmente modelable
sales_modelable[["sales", "onpromotion"]].describe()

,sales,onpromotion
count,2.697207e+06,2.697207e+06
mean,3.980580e+02,2.895819e+00
std,1.155463e+03,1.285544e+01
min,0.000000e+00,0.000000e+00
25%,1.000000e+00,0.000000e+00
50%,1.900000e+01,0.000000e+00
75%,2.408610e+02,0.000000e+00
max,1.247170e+05,7.410000e+02


In [15]:
total_modelable = len(sales_modelable)
zero_modelable = (sales_modelable["sales"] == 0).sum()
zero_pct_modelable = zero_modelable / total_modelable * 100

print(f"Registros modelables: {total_modelable:,}")
print(f"Tiendas: {sales_modelable['store_nbr'].nunique()}")
print(f"Familias: {sales_modelable['family'].nunique()}")
print(f"Ceros reales de demanda: {zero_modelable:,}")
print(f"Porcentaje de ceros reales: {zero_pct_modelable:.2f}%")
print(f"Ventas negativas: {(sales_modelable['sales'] < 0).sum():,}")

Registros modelables: 2,697,207
Tiendas: 54
Familias: 33
Ceros reales de demanda: 635,449
Porcentaje de ceros reales: 23.56%
Ventas negativas: 0


In [16]:
zero_comparison = pd.DataFrame({
    "panel": [
        "Original",
        "Operativo",
        "Modelable"
    ],
    "registros": [
        len(sales),
        len(sales_valid),
        len(sales_modelable)
    ],
    "ceros": [
        (sales["sales"] == 0).sum(),
        (sales_valid["sales"] == 0).sum(),
        (sales_modelable["sales"] == 0).sum()
    ]
})

zero_comparison["porcentaje_ceros"] = (
    zero_comparison["ceros"]
    / zero_comparison["registros"]
    * 100
)

zero_comparison

,panel,registros,ceros,porcentaje_ceros
0,Original,3000888,939130,31.295070
1,Operativo,2778831,717073,25.804844
2,Modelable,2697207,635449,23.559519


In [17]:
fig = px.bar(
    zero_comparison,
    x="panel",
    y="porcentaje_ceros",
    text="porcentaje_ceros",
    title="Impacto de los ceros estructurales sobre el panel"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Panel",
    yaxis_title="% de registros con sales = 0"
)

fig.show()

In [18]:
sales_modelable["sales"].quantile([
    0,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1
])

0.00         0.000
0.25         1.000
0.50        19.000
0.75       240.861
0.90       995.000
0.95      2176.000
0.99      5807.000
1.00    124717.000
Name: sales, dtype: float64

### Distribución de ventas (muestra)

In [19]:
sales_sample = sales_modelable.sample(
    n=min(200_000, len(sales_modelable)),
    random_state=42
)

p99 = sales_modelable["sales"].quantile(0.99)

sales_visual = sales_sample[
    sales_sample["sales"] <= p99
]

In [20]:
fig = px.histogram(
    sales_visual,
    x="sales",
    nbins=100,
    title="Distribución de ventas — panel modelable (hasta P99)"
)

fig.update_layout(
    xaxis_title="Ventas",
    yaxis_title="Cantidad de registros"
)

fig.show()

In [21]:
zero_sales_by_family = (
    sales_modelable
    .assign(is_zero=sales_modelable["sales"] == 0)
    .groupby("family", as_index=False)
    .agg(
        registros=("sales", "size"),
        ventas_cero=("is_zero", "sum")
    )
)

zero_sales_by_family["pct_zero"] = (
    zero_sales_by_family["ventas_cero"]
    / zero_sales_by_family["registros"]
    * 100
)

zero_sales_by_family = (
    zero_sales_by_family
    .sort_values("pct_zero", ascending=False)
)

zero_sales_by_family.head(15)

,family,registros,ventas_cero,pct_zero
4,BOOKS,42289,39520,93.452198
1,BABY CARE,65576,60239,91.861352
31,SCHOOL AND OFFICE SUPPLIES,84207,60639,72.011828
17,HOME APPLIANCES,84207,60125,71.401427
23,MAGAZINES,84207,44685,53.065660
26,PET SUPPLIES,84207,42936,50.988635
19,LADIESWEAR,68181,31667,46.445491
14,HARDWARE,84207,36795,43.695892
27,PLAYERS AND ELECTRONICS,84207,34089,40.482383
6,CELEBRATION,84207,32950,39.129764


In [22]:
fig = px.bar(
    zero_sales_by_family.head(15),
    x="pct_zero",
    y="family",
    orientation="h",
    title="Familias con mayor proporción de ceros reales de demanda"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="% de ceros reales",
    yaxis_title="Familia"
)

fig.show()

### Hallazgo — estructura de los ceros

El análisis estructural muestra que los registros con `sales = 0` no representan un único fenómeno.

En el panel original, el 31,30 % de las observaciones presentaban ventas iguales a cero. Sin embargo, se identificaron **222.057 registros pre-apertura**, equivalentes al **7,40 % del dataset**, y **81.624 registros correspondientes a familias que determinadas tiendas no comercializan**.

Luego de excluir estos ceros estructurales, el panel modelable queda compuesto por **2.697.207 observaciones**, de las cuales **635.449 presentan ausencia real de ventas**, equivalente al **23,56 % del panel modelable**.

La demanda cero tampoco se distribuye uniformemente entre categorías: familias como `BOOKS` y `BABY CARE` presentan más del **90 % de registros sin ventas**, evidenciando un comportamiento altamente intermitente.

Esta separación evita que las etapas posteriores interpreten como baja demanda situaciones en las que la tienda todavía no estaba operativa o la familia de productos no formaba parte de su surtido.

## 6.2 Evolución temporal de las ventas

In [23]:
sales_daily = (
    sales
    .groupby("date", as_index=False)["sales"]
    .sum()
)

sales_daily.head()

,date,sales
0,2013-01-01,2511.618999
1,2013-01-02,496092.417944
2,2013-01-03,361461.231124
3,2013-01-04,354459.677093
4,2013-01-05,477350.121229


In [24]:
fig = px.line(
    sales_daily,
    x="date",
    y="sales",
    title="Evolución diaria de las ventas totales"
)

fig.show()

In [25]:
sales_monthly = (
    sales
    .set_index("date")
    .resample("ME")["sales"]
    .sum()
    .reset_index()
)

fig = px.line(
    sales_monthly,
    x="date",
    y="sales",
    title="Evolución mensual de las ventas totales"
)

fig.show()

In [26]:
print(sales["date"].max())

sales[
    sales["date"].dt.to_period("M") == "2017-08"
]["date"].nunique()

2017-08-15 00:00:00


15

Conclusión temporal: Se observa una tendencia creciente de las ventas a lo largo del período analizado, acompañada por fluctuaciones y patrones recurrentes. La caída observada en agosto de 2017 no representa una disminución real de la demanda, ya que el dataset finaliza el 15 de agosto y, por lo tanto, dicho mes contiene información parcial.

### Comparación mensual sin agosto de 2017 parcial

In [27]:
sales_monthly_complete = sales_monthly[
    sales_monthly["date"] < "2017-08-01"
]

fig = px.line(
    sales_monthly_complete,
    x="date",
    y="sales",
    title="Evolución mensual de ventas — meses completos"
)

fig.show()

## 6.3 Feriados y eventos — análisis preliminar

In [28]:
national_holidays = holidays[
    (holidays["locale"] == "National") &
    (holidays["transferred"] == False)
][["date", "description"]].drop_duplicates("date")

In [29]:
sales_holiday = sales_daily.merge(
    national_holidays,
    on="date",
    how="left"
)

sales_holiday["is_holiday"] = sales_holiday["description"].notna()

sales_holiday.head()

,date,sales,description,is_holiday
0,2013-01-01,2511.618999,Primer dia del ano,True
1,2013-01-02,496092.417944,NaN,False
2,2013-01-03,361461.231124,NaN,False
3,2013-01-04,354459.677093,NaN,False
4,2013-01-05,477350.121229,Recupero puente Navidad,True


In [30]:
sales_holiday.groupby("is_holiday")["sales"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
is_holiday,,,
False,1548,627918.111596,624754.402274
True,136,747262.613625,742923.217484


In [31]:
sales_holiday[
    sales_holiday["date"].dt.strftime("%m-%d") == "01-01"
]

,date,sales,description,is_holiday
0,2013-01-01,2511.618999,Primer dia del ano,True
364,2014-01-01,8602.065404,Primer dia del ano,True
728,2015-01-01,12773.616980,Primer dia del ano,True
1092,2016-01-01,16433.394000,Primer dia del ano,True
1457,2017-01-01,12082.500997,NaN,False


Impacto de feriados: Los feriados no presentan un efecto uniforme sobre las ventas. En promedio, los feriados nacionales analizados registran mayores ventas que los días normales. Sin embargo, determinados eventos presentan comportamientos particulares; el 1 de enero muestra una caída extrema y recurrente de las ventas, evidenciando la necesidad de considerar el tipo específico de feriado y no únicamente una variable binaria de feriado/no feriado.

## 6.4 Ventas por familia de producto

In [32]:
# Ventas totales por familia
sales_by_family = (
    sales
    .groupby("family", as_index=False)["sales"]
    .sum()
    .sort_values("sales", ascending=False)
)

sales_by_family.head(10)

,family,sales
12,GROCERY I,3.434627e+08
3,BEVERAGES,2.169545e+08
30,PRODUCE,1.227047e+08
7,CLEANING,9.752129e+07
8,DAIRY,6.448771e+07
5,BREAD/BAKERY,4.213395e+07
28,POULTRY,3.187600e+07
24,MEATS,3.108647e+07
25,PERSONAL CARE,2.459205e+07
9,DELI,2.411032e+07


In [33]:
fig = px.bar(
    sales_by_family.head(15),
    x="sales",
    y="family",
    orientation="h",
    title="Top 15 familias por ventas totales"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [34]:
zero_sales_by_family = (
    sales
    .assign(is_zero=sales["sales"] == 0)
    .groupby("family", as_index=False)
    .agg(
        registros=("sales", "size"),
        ventas_cero=("is_zero", "sum")
    )
)

zero_sales_by_family["pct_zero"] = (
    zero_sales_by_family["ventas_cero"]
    / zero_sales_by_family["registros"]
    * 100
)

zero_sales_by_family = zero_sales_by_family.sort_values(
    "pct_zero",
    ascending=False
)

zero_sales_by_family.head(10)

,family,registros,ventas_cero,pct_zero
4,BOOKS,90936,88167,96.955001
1,BABY CARE,90936,85599,94.131037
31,SCHOOL AND OFFICE SUPPLIES,90936,67368,74.082871
17,HOME APPLIANCES,90936,66854,73.517639
19,LADIESWEAR,90936,54422,59.846485
23,MAGAZINES,90936,51414,56.538665
26,PET SUPPLIES,90936,49665,54.615334
14,HARDWARE,90936,43524,47.862233
20,LAWN AND GARDEN,90936,42544,46.784552
27,PLAYERS AND ELECTRONICS,90936,40818,44.886514


In [35]:
fig = px.bar(
    zero_sales_by_family.head(15),
    x="pct_zero",
    y="family",
    orientation="h",
    title="Familias con mayor porcentaje de registros sin ventas"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

Conclusión por familia: La demanda se encuentra fuertemente concentrada en pocas familias de productos, especialmente GROCERY I, BEVERAGES y PRODUCE. Al mismo tiempo, existen familias con una elevada proporción de registros sin ventas, lo que evidencia patrones de demanda intermitente. Esto deberá considerarse posteriormente tanto en la ingeniería de variables como en la construcción del modelo predictivo.

## 6.5 Ventas por tienda

In [36]:
sales_by_store = (
    sales
    .groupby("store_nbr", as_index=False)["sales"]
    .sum()
    .sort_values("sales", ascending=False)
)

sales_by_store.head(10)

,store_nbr,sales
43,44,6.208755e+07
44,45,5.449801e+07
46,47,5.094831e+07
2,3,5.048191e+07
48,49,4.342010e+07
45,46,4.189606e+07
47,48,3.593313e+07
50,51,3.291149e+07
7,8,3.049429e+07
49,50,2.865302e+07


In [37]:
fig = px.bar(
    sales_by_store.head(15),
    x="sales",
    y="store_nbr",
    orientation="h",
    title="Top 15 tiendas por ventas totales"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [38]:
sales_by_store = sales_by_store.merge(
    stores,
    on="store_nbr",
    how="left"
)

sales_by_store.head()

,store_nbr,sales,city,state,type,cluster
0,44,6.208755e+07,Quito,Pichincha,A,5
1,45,5.449801e+07,Quito,Pichincha,A,11
2,47,5.094831e+07,Quito,Pichincha,A,14
3,3,5.048191e+07,Quito,Pichincha,D,8
4,49,4.342010e+07,Quito,Pichincha,A,11


In [39]:
sales_by_type = (
    sales_by_store
    .groupby("type", as_index=False)["sales"]
    .sum()
    .sort_values("sales", ascending=False)
)

fig = px.bar(
    sales_by_type,
    x="type",
    y="sales",
    title="Ventas totales por tipo de tienda"
)

fig.show()

In [40]:
sales_store_type = (
    sales_by_store
    .groupby("type", as_index=False)
    .agg(
        stores=("store_nbr", "count"),
        total_sales=("sales", "sum"),
        avg_sales_per_store=("sales", "mean"),
        median_sales_per_store=("sales", "median")
    )
)

sales_store_type

,type,stores,total_sales,avg_sales_per_store,median_sales_per_store
0,A,9,3.530438e+08,3.922709e+07,4.189606e+07
1,B,8,1.452606e+08,1.815758e+07,1.741880e+07
2,C,15,1.644347e+08,1.096232e+07,1.098641e+07
3,D,18,3.510833e+08,1.950463e+07,1.888485e+07
4,E,4,5.982244e+07,1.495561e+07,1.585706e+07


In [41]:
fig = px.bar(
    sales_store_type,
    x="type",
    y="avg_sales_per_store",
    title="Ventas promedio por tienda según tipo"
)

fig.show()

In [42]:
top_stores = sales_by_store.head(15).copy()
top_stores["store_nbr"] = top_stores["store_nbr"].astype(str)

fig = px.bar(
    top_stores,
    x="sales",
    y="store_nbr",
    orientation="h",
    title="Top 15 tiendas por ventas totales"
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

Conclusión por tienda: Las ventas presentan diferencias significativas entre establecimientos. El tipo A destaca por registrar el mayor volumen promedio de ventas por tienda, mientras que el tipo D alcanza ventas totales elevadas principalmente por contar con una mayor cantidad de establecimientos. Esto indica que características como el tipo y la identidad de la tienda pueden aportar información relevante para explicar la demanda.

## 6.6 Promociones y ventas — efecto controlado

El análisis inicial comparaba directamente todos los registros con y sin promoción. Sin embargo, `onpromotion` presenta una fuerte dependencia temporal y las promociones se concentran en determinadas familias de mayor volumen.

Para reducir este sesgo, se analiza primero la evolución temporal de las promociones y posteriormente se compara su efecto desde 2014 dentro de combinaciones tienda–familia.

In [43]:
promo_year = (
    sales_modelable
    .assign(
        year=sales_modelable["date"].dt.year,
        has_promotion=sales_modelable["onpromotion"] > 0
    )
    .groupby("year", as_index=False)
    .agg(
        registros=("sales", "size"),
        registros_promo=("has_promotion", "sum"),
        ventas_promedio=("sales", "mean")
    )
)

promo_year["pct_promocion"] = (
    promo_year["registros_promo"]
    / promo_year["registros"]
    * 100
)

promo_year

,year,registros,registros_promo,ventas_promedio,pct_promocion
0,2013,542283,0,258.940468,0.000000
1,2014,554948,64979,377.466441,11.709025
2,2015,591102,135302,407.510211,22.889789
3,2016,619770,227981,465.744587,36.784775
4,2017,389104,183067,499.139223,47.048347


In [44]:
fig = px.bar(
    promo_year,
    x="year",
    y="pct_promocion",
    text="pct_promocion",
    title="Evolución de la presencia de promociones por año"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Año",
    yaxis_title="% de registros con promoción"
)

fig.show()

In [45]:
promo_data = sales_modelable[
    sales_modelable["date"] >= "2014-01-01"
].copy()

promo_data["has_promotion"] = (
    promo_data["onpromotion"] > 0
)

In [46]:
no_promo = (
    promo_data[
        ~promo_data["has_promotion"]
    ]
    .groupby(
        ["store_nbr", "family"],
        as_index=False
    )
    .agg(
        n_sin_promo=("sales", "size"),
        ventas_sin_promo=("sales", "mean")
    )
)

In [47]:
with_promo = (
    promo_data[
        promo_data["has_promotion"]
    ]
    .groupby(
        ["store_nbr", "family"],
        as_index=False
    )
    .agg(
        n_con_promo=("sales", "size"),
        ventas_con_promo=("sales", "mean")
    )
)

In [48]:
promo_lift = no_promo.merge(
    with_promo,
    on=["store_nbr", "family"],
    how="inner"
)

In [49]:
promo_lift = promo_lift[
    (promo_lift["n_sin_promo"] >= 20) &
    (promo_lift["n_con_promo"] >= 20) &
    (promo_lift["ventas_sin_promo"] > 0)
].copy()

In [50]:
promo_lift["lift_ratio"] = (
    promo_lift["ventas_con_promo"]
    / promo_lift["ventas_sin_promo"]
)

In [51]:
family_promo_lift = (
    promo_lift
    .groupby("family", as_index=False)
    .agg(
        tiendas_comparables=("store_nbr", "nunique"),
        lift_mediano=("lift_ratio", "median")
    )
    .sort_values(
        "lift_mediano",
        ascending=False
    )
)

family_promo_lift

,family,tiendas_comparables,lift_mediano
29,SCHOOL AND OFFICE SUPPLIES,45,21.997562
1,BABY CARE,1,8.488806
21,MAGAZINES,2,2.540049
13,HARDWARE,2,2.294542
28,PRODUCE,53,1.850684
15,HOME AND KITCHEN II,53,1.825217
2,BEAUTY,54,1.754604
24,PET SUPPLIES,35,1.750712
10,FROZEN FOODS,53,1.681101
16,HOME CARE,53,1.665221


In [52]:
familias_principales = [
    "PRODUCE",
    "BEVERAGES",
    "DAIRY",
    "GROCERY I",
    "MEATS"
]

family_promo_lift[
    family_promo_lift["family"].isin(
        familias_principales
    )
].sort_values(
    "lift_mediano",
    ascending=False
)

,family,tiendas_comparables,lift_mediano
28,PRODUCE,53,1.850684
3,BEVERAGES,52,1.541247
11,GROCERY I,50,1.239632
22,MEATS,54,1.183585
7,DAIRY,53,1.140558


In [53]:
promo_plot = (
    family_promo_lift[
        family_promo_lift["family"].isin(
            familias_principales
        )
    ]
    .sort_values(
        "lift_mediano",
        ascending=True
    )
)

fig = px.bar(
    promo_plot,
    x="lift_mediano",
    y="family",
    orientation="h",
    text="lift_mediano",
    title="Lift promocional dentro de tienda–familia desde 2014"
)

fig.update_traces(
    texttemplate="%{text:.2f}×",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Ratio ventas con promoción / sin promoción",
    yaxis_title="Familia"
)

fig.show()

### Hallazgo — efecto promocional controlado

La comparación global entre registros con y sin promoción estaba fuertemente afectada por la evolución temporal de la variable `onpromotion`. Mientras que en 2013 no se registran promociones, su presencia aumenta progresivamente hasta alcanzar aproximadamente el **47,0 % de los registros en 2017**.

Para reducir este sesgo, el efecto promocional se analizó desde 2014 y dentro de combinaciones comparables tienda–familia.

En las principales familias, el lift promocional mediano obtenido fue de **1,85× en PRODUCE**, **1,54× en BEVERAGES**, **1,24× en GROCERY I**, **1,18× en MEATS** y **1,14× en DAIRY**.

Esto indica que las promociones continúan asociándose con mayores niveles de ventas, pero el efecto es considerablemente menor que el observado en la comparación global inicial y varía de forma importante entre familias.

Por lo tanto, la promoción debe analizarse en conjunto con la familia, la tienda y el período temporal, evitando interpretar la diferencia global como un efecto causal directo.

## 6.7 Estacionalidad de la demanda

La demanda se analiza como una serie temporal para identificar patrones recurrentes asociados al calendario.

Se estudian tres componentes principales: día de la semana, día del mes y la interacción entre día de semana y mes del año.

Los días 1 de enero se excluyen de este análisis estacional debido a que representan días de cierre o actividad excepcionalmente baja y no un patrón normal de demanda.

In [54]:
daily_modelable = (
    sales_modelable
    .groupby("date", as_index=False)["sales"]
    .sum()
)

daily_modelable["day_of_week"] = daily_modelable["date"].dt.dayofweek
daily_modelable["day_name"] = daily_modelable["date"].dt.day_name()
daily_modelable["day_of_month"] = daily_modelable["date"].dt.day
daily_modelable["month"] = daily_modelable["date"].dt.month
daily_modelable["year"] = daily_modelable["date"].dt.year

# Excluir 1 de enero: cierre / funcionamiento excepcional
seasonal_daily = daily_modelable[
    ~(
        (daily_modelable["date"].dt.month == 1) &
        (daily_modelable["date"].dt.day == 1)
    )
].copy()

In [55]:
overall_daily_mean = seasonal_daily["sales"].mean()

weekday_index = (
    seasonal_daily
    .groupby(
        ["day_of_week", "day_name"],
        as_index=False
    )
    .agg(
        ventas_promedio=("sales", "mean")
    )
)

weekday_index["indice"] = (
    weekday_index["ventas_promedio"]
    / overall_daily_mean
)

weekday_index = weekday_index.sort_values("day_of_week")

weekday_index

,day_of_week,day_name,ventas_promedio,indice
0,0,Monday,617542.713052,0.965780
1,1,Tuesday,572280.504806,0.894994
2,2,Wednesday,595690.756351,0.931606
3,3,Thursday,507329.852094,0.793417
4,4,Friday,581930.599441,0.910086
5,5,Saturday,772205.593943,1.207659
6,6,Sunday,828620.362670,1.295886


In [56]:
fig = px.bar(
    weekday_index,
    x="day_name",
    y="indice",
    text="indice",
    title="Índice de demanda por día de la semana"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.add_hline(
    y=1,
    line_dash="dash"
)

fig.update_layout(
    xaxis_title="Día de la semana",
    yaxis_title="Índice vs. día promedio"
)

fig.show()

In [57]:
weekday_baseline = (
    seasonal_daily
    .groupby("day_of_week")["sales"]
    .mean()
)

seasonal_daily["weekday_baseline"] = (
    seasonal_daily["day_of_week"]
    .map(weekday_baseline)
)

In [58]:
seasonal_daily["sales_weekday_adjusted"] = (
    seasonal_daily["sales"]
    / seasonal_daily["weekday_baseline"]
)

In [59]:
day_month_index = (
    seasonal_daily
    .groupby("day_of_month", as_index=False)
    .agg(
        indice=("sales_weekday_adjusted", "mean")
    )
)

day_month_index

,day_of_month,indice
0,1,1.224868
1,2,1.174979
2,3,1.135367
3,4,1.095084
4,5,1.057656
5,6,1.028931
6,7,1.011867
7,8,0.981095
8,9,0.976827
9,10,0.957741


In [60]:
fig = px.line(
    day_month_index,
    x="day_of_month",
    y="indice",
    markers=True,
    title="Índice de demanda por día del mes — ajustado por día de semana"
)

fig.add_hline(
    y=1,
    line_dash="dash"
)

# Días relevantes
for day in [1, 16, 31]:
    fig.add_vline(
        x=day,
        line_dash="dot"
    )

fig.update_layout(
    xaxis_title="Día del mes",
    yaxis_title="Índice de demanda ajustado"
)

fig.show()

In [61]:
seasonal_daily["year_month"] = (
    seasonal_daily["date"]
    .dt.to_period("M")
)

seasonal_daily["monthly_mean"] = (
    seasonal_daily
    .groupby("year_month")["sales"]
    .transform("mean")
)

seasonal_daily["monthly_index"] = (
    seasonal_daily["sales"]
    / seasonal_daily["monthly_mean"]
)

In [62]:
heatmap_data = (
    seasonal_daily
    .groupby(
        ["month", "day_of_week"],
        as_index=False
    )
    .agg(
        indice=("monthly_index", "mean")
    )
)

In [63]:
heatmap_pivot = heatmap_data.pivot(
    index="day_of_week",
    columns="month",
    values="indice"
)

day_labels = [
    "Lunes",
    "Martes",
    "Miércoles",
    "Jueves",
    "Viernes",
    "Sábado",
    "Domingo"
]

month_labels = [
    "Ene", "Feb", "Mar", "Abr",
    "May", "Jun", "Jul", "Ago",
    "Sep", "Oct", "Nov", "Dic"
]

heatmap_pivot.index = day_labels
heatmap_pivot.columns = month_labels

In [64]:
fig = px.imshow(
    heatmap_pivot,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Estacionalidad: día de semana × mes"
)

fig.update_layout(
    xaxis_title="Mes",
    yaxis_title="Día de la semana"
)

fig.show()

### Hallazgo — estacionalidad de la demanda

El análisis temporal evidencia una marcada estacionalidad asociada al calendario.

A nivel semanal, el **domingo presenta un índice de demanda de 1,30** y el **sábado de 1,21**, mientras que el **jueves desciende hasta 0,79**. Esto representa una diferencia relativa cercana al **63 % entre el día de mayor y menor demanda**, confirmando que el día de la semana constituye una señal predictiva relevante.

Luego de ajustar el efecto del día de semana, también aparece un patrón claro dentro del mes. El **día 1 registra un índice de 1,22**, seguido por valores elevados en los días 2 y 3. Posteriormente la demanda disminuye progresivamente, presenta un pequeño repunte alrededor del **día 16** y alcanza sus niveles más bajos entre los **días 25 y 28**, con índices cercanos a 0,91–0,93. Hacia el cierre del mes vuelve a observarse una recuperación.

Este comportamiento es compatible con un efecto de calendario asociado a comienzos de mes, quincena y períodos previos al cobro.

Finalmente, el heatmap día de semana × mes muestra que el patrón semanal es estable durante gran parte del año: los fines de semana, especialmente los domingos, presentan los mayores niveles relativos de demanda, mientras que los jueves muestran sistemáticamente los niveles más bajos.

Estos resultados justifican incorporar posteriormente variables de calendario como día de semana, día del mes, fin de semana y posición dentro del ciclo mensual.

## 6.8 Transacciones y ventas

Se analiza la relación entre el volumen diario de transacciones y las ventas agregadas por tienda.

Dado que ambas variables presentan tendencia y estacionalidad, la relación no se evalúa únicamente en niveles. También se estudian primeras diferencias y rezagos temporales para distinguir una asociación contemporánea de una posible relación predictiva.

In [65]:
# Ventas diarias por tienda
sales_store_daily = (
    sales_valid
    .groupby(["date", "store_nbr"], as_index=False)["sales"]
    .sum()
)

sales_transactions = (
    sales_store_daily
    .merge(
        transactions,
        on=["date", "store_nbr"],
        how="inner"
    )
    .sort_values(["store_nbr", "date"])
)

In [66]:
corr_tx_levels = (
    sales_transactions["sales"]
    .corr(sales_transactions["transactions"])
)

print(
    f"Correlación en niveles: "
    f"{corr_tx_levels:.3f}"
)

Correlación en niveles: 0.837


In [67]:
sales_transactions["gap_days"] = (
    sales_transactions
    .groupby("store_nbr")["date"]
    .diff()
    .dt.days
)

sales_transactions["diff_sales"] = (
    sales_transactions
    .groupby("store_nbr")["sales"]
    .diff()
)

sales_transactions["diff_transactions"] = (
    sales_transactions
    .groupby("store_nbr")["transactions"]
    .diff()
)

tx_diff = sales_transactions[
    sales_transactions["gap_days"] == 1
].dropna(
    subset=["diff_sales", "diff_transactions"]
).copy()

In [68]:
corr_tx_diff = (
    tx_diff["diff_sales"]
    .corr(tx_diff["diff_transactions"])
)

print(
    f"Correlación primeras diferencias: "
    f"{corr_tx_diff:.3f}"
)

Correlación primeras diferencias: 0.783


In [69]:
tx_results = [
    {
        "comparacion": "Niveles",
        "correlacion": corr_tx_levels
    },
    {
        "comparacion": "Δ mismo día",
        "correlacion": corr_tx_diff
    }
]

for lag in [1, 7]:

    lagged = tx_diff[
        ["date", "store_nbr", "diff_transactions"]
    ].copy()

    lagged["date"] = (
        lagged["date"]
        + pd.Timedelta(days=lag)
    )

    lagged = lagged.rename(
        columns={
            "diff_transactions":
            f"diff_transactions_lag{lag}"
        }
    )

    comparison = tx_diff.merge(
        lagged,
        on=["date", "store_nbr"],
        how="inner"
    )

    corr = comparison["diff_sales"].corr(
        comparison[f"diff_transactions_lag{lag}"]
    )

    tx_results.append({
        "comparacion": f"Δ transacciones t-{lag}",
        "correlacion": corr
    })

tx_corr_summary = pd.DataFrame(tx_results)

tx_corr_summary

,comparacion,correlacion
0,Niveles,0.837384
1,Δ mismo día,0.783366
2,Δ transacciones t-1,0.032860
3,Δ transacciones t-7,0.604634


In [70]:
fig = px.bar(
    tx_corr_summary,
    x="comparacion",
    y="correlacion",
    text="correlacion",
    title="Transacciones y ventas — niveles, diferencias y rezagos"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Comparación",
    yaxis_title="Correlación"
)

fig.show()

In [71]:
sales_transactions["transaction_group"] = pd.qcut(
    sales_transactions["transactions"],
    q=10,
    duplicates="drop"
)

transaction_summary = (
    sales_transactions
    .groupby("transaction_group", observed=True)
    .agg(
        avg_transactions=("transactions", "mean"),
        avg_sales=("sales", "mean")
    )
    .reset_index()
)

transaction_summary

,transaction_group,avg_transactions,avg_sales
0,"(4.999, 754.0]",637.218301,4285.888982
1,"(754.0, 961.0]",861.172039,6330.227792
2,"(961.0, 1125.0]",1046.064273,7776.462341
3,"(1125.0, 1260.0]",1194.286366,8463.891359
4,"(1260.0, 1393.0]",1325.175687,9787.459895
5,"(1393.0, 1590.0]",1482.336604,11545.794273
6,"(1590.0, 1873.0]",1730.951976,13652.240683
7,"(1873.0, 2341.6]",2089.134862,14996.462156
8,"(2341.6, 3071.0]",2676.772444,18692.063860
9,"(3071.0, 8359.0]",3907.712608,32819.397785


In [72]:
fig = px.line(
    transaction_summary,
    x="avg_transactions",
    y="avg_sales",
    markers=True,
    title="Ventas promedio según nivel de transacciones"
)

fig.show()

### Hallazgo — transacciones y ventas

En niveles, las ventas diarias por tienda presentan una correlación elevada con el número de transacciones (**r = 0,837**). Para verificar que esta asociación no estuviera explicada únicamente por tendencias compartidas, se repitió el análisis utilizando primeras diferencias.

La correlación entre los cambios diarios de ventas y transacciones se mantiene elevada (**r = 0,783**), indicando que ambas variables presentan una relación contemporánea importante incluso después de remover parte de su tendencia.

Al analizar rezagos, el cambio de transacciones del día anterior presenta una correlación prácticamente nula con el cambio de ventas actual (**r = 0,033**), mientras que el rezago de siete días conserva una asociación considerable (**r = 0,605**). Este resultado es consistente con la marcada estacionalidad semanal observada previamente.

Por lo tanto, las transacciones contienen información relevante sobre la actividad comercial, aunque las transacciones del mismo día no pueden utilizarse directamente para predecir ventas de ese mismo día si todavía no son conocidas al momento del pronóstico. En etapas posteriores deberán considerarse únicamente valores históricos o variables derivadas mediante rezagos.

## 6.9 Petróleo y ventas

El precio del petróleo se incorpora como variable externa de contexto macroeconómico.

Para evitar conclusiones espurias derivadas de tendencias compartidas, se compara su relación con las ventas tanto en niveles como en primeras diferencias y rezagos temporales.

Además, el precio se trata respetando el orden temporal de la información, utilizando únicamente valores conocidos hasta cada fecha.

In [80]:
sales_daily_corr = (
    sales_valid
    .groupby("date", as_index=False)["sales"]
    .sum()
)

sales_oil_corr = (
    sales_daily_corr
    .merge(
        oil,
        on="date",
        how="left"
    )
    .sort_values("date")
)

In [81]:
corr_oil_levels = (
    sales_oil_corr["sales"]
    .corr(sales_oil_corr["dcoilwtico"])
)

print(
    f"Correlación en niveles: "
    f"{corr_oil_levels:.3f}"
)

Correlación en niveles: -0.627


In [82]:
sales_oil_corr["gap_days"] = (
    sales_oil_corr["date"]
    .diff()
    .dt.days
)

sales_oil_corr["diff_sales"] = (
    sales_oil_corr["sales"].diff()
)

sales_oil_corr["diff_oil"] = (
    sales_oil_corr["dcoilwtico"].diff()
)

oil_diff = sales_oil_corr[
    sales_oil_corr["gap_days"] == 1
].dropna(
    subset=["diff_sales", "diff_oil"]
).copy()

In [83]:
corr_oil_diff = (
    oil_diff["diff_sales"]
    .corr(oil_diff["diff_oil"])
)

print(
    f"Correlación primeras diferencias: "
    f"{corr_oil_diff:.3f}"
)

Correlación primeras diferencias: 0.025


In [84]:
oil_results = [
    {
        "comparacion": "Niveles",
        "correlacion": corr_oil_levels
    },
    {
        "comparacion": "Δ mismo día",
        "correlacion": corr_oil_diff
    }
]

for lag in [1, 7]:

    lagged = oil_diff[
        ["date", "diff_oil"]
    ].copy()

    lagged["date"] = (
        lagged["date"]
        + pd.Timedelta(days=lag)
    )

    lagged = lagged.rename(
        columns={
            "diff_oil":
            f"diff_oil_lag{lag}"
        }
    )

    comparison = oil_diff.merge(
        lagged,
        on="date",
        how="inner"
    )

    corr = comparison["diff_sales"].corr(
        comparison[f"diff_oil_lag{lag}"]
    )

    oil_results.append({
        "comparacion": f"Δ petróleo t-{lag}",
        "correlacion": corr
    })

oil_corr_summary = pd.DataFrame(oil_results)

oil_corr_summary

,comparacion,correlacion
0,Niveles,-0.626856
1,Δ mismo día,0.024752
2,Δ petróleo t-1,0.014481
3,Δ petróleo t-7,0.021046


In [85]:
fig = px.bar(
    oil_corr_summary,
    x="comparacion",
    y="correlacion",
    text="correlacion",
    title="Petróleo y ventas — niveles, diferencias y rezagos"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Comparación",
    yaxis_title="Correlación"
)

fig.show()

In [87]:
oil_time = sales_oil_corr.copy()

oil_time["sales_norm"] = (
    oil_time["sales"] / oil_time["sales"].max()
)

oil_time["oil_norm"] = (
    oil_time["dcoilwtico"] / oil_time["dcoilwtico"].max()
)

fig = px.line(
    oil_time,
    x="date",
    y=["sales_norm", "oil_norm"],
    title="Evolución relativa de ventas y precio del petróleo"
)

fig.show()

### Hallazgo — petróleo y ventas

Se analizó la relación entre el precio diario del petróleo y las ventas agregadas por fecha.

En niveles, ambas series presentan una correlación negativa moderada (**r = -0.627**). Sin embargo, este resultado puede estar influido por tendencias de largo plazo presentes en ambas variables y no necesariamente por una relación operativa directa.

Para verificarlo, se repitió el análisis utilizando primeras diferencias diarias. En este caso, la correlación entre cambios diarios de ventas y cambios diarios del precio del petróleo resulta prácticamente nula (**r = 0.025**).

También se evaluaron rezagos del petróleo. Tanto el rezago de un día (**r = 0.014**) como el de siete días (**r = 0.021**) muestran asociaciones despreciables con las variaciones de ventas.

En conjunto, estos resultados indican que el precio del petróleo no presenta una relación útil de corto plazo con la demanda observada en esta etapa del análisis. Por lo tanto, su aporte predictivo potencial sería limitado, al menos bajo estas transformaciones simples.

### Tratamiento temporal del precio del petróleo

El precio del petróleo se reindexó previamente sobre un calendario diario completo.

Los días sin cotización utilizan únicamente el último precio conocido mediante `forward fill (ffill)`. De esta forma, para cada fecha sólo se utiliza información que ya estaba disponible en ese momento, evitando fuga temporal.

No se utiliza interpolación lineal ni `backfill`, ya que ambos métodos podrían incorporar información procedente de fechas futuras.

El primer registro del período permanece sin imputar al no existir un precio previo conocido. Se conserva como valor faltante para evitar incorporar información futura.

## 6.10 Outliers y anomalías temporales

La detección de anomalías se realiza sobre la serie temporal diaria.

El método IQR global utilizado inicialmente no resulta adecuado porque las ventas presentan tendencia y estacionalidad. Por este motivo se utiliza una referencia móvil basada en mediana y MAD, que compara cada día con su entorno temporal reciente.

In [ ]:
daily_anomaly = (
    sales_modelable
    .groupby("date", as_index=False)["sales"]
    .sum()
    .set_index("date")
    .reindex(
        pd.date_range(
            sales_modelable["date"].min(),
            sales_modelable["date"].max(),
            freq="D"
        )
    )
    .rename_axis("date")
)

daily_anomaly.head()

,sales
date,
2013-01-01,2511.618999
2013-01-02,496092.417944
2013-01-03,361461.231124
2013-01-04,354459.677093
2013-01-05,477350.121229


In [ ]:
rolling = daily_anomaly["sales"].rolling(
    window=28,
    center=True,
    min_periods=14
)

rolling_median = rolling.median()

In [ ]:
# 1. MAD corregido
rolling_mad = rolling.apply(
    lambda x: np.nanmedian(
        np.abs(x - np.nanmedian(x))
    ),
    raw=True
)

# 2. Recalcular robust_z
daily_anomaly["rolling_median"] = rolling_median

daily_anomaly["robust_z"] = (
    daily_anomaly["sales"]
    - daily_anomaly["rolling_median"]
) / (
    1.4826 * rolling_mad
)

# 3. Volver a detectar anomalías
anomalies = daily_anomaly[
    daily_anomaly["robust_z"].abs() > 4
].copy()

print(f"Anomalías detectadas: {len(anomalies)}")

anomalies[
    ["sales", "rolling_median", "robust_z"]
].sort_values("robust_z")

Anomalías detectadas: 21


,sales,rolling_median,robust_z
date,,,
2014-01-01,8.602065e+03,5.634506e+05,-5.220684
2016-01-01,1.643339e+04,9.467252e+05,-4.240582
2017-01-01,1.208250e+04,1.028243e+06,-4.171557
2013-01-01,2.511619e+03,3.452912e+05,-4.073302
2013-05-18,4.809696e+05,3.313676e+05,4.061484
2017-07-02,1.296379e+06,7.967098e+05,4.145092
2017-05-01,1.306699e+06,7.809788e+05,4.211608
2017-01-22,1.148214e+06,7.804392e+05,4.222725
2013-02-02,5.188875e+05,3.154046e+05,4.276727


In [ ]:
jan1_check = daily_anomaly[
    (daily_anomaly.index.month == 1) &
    (daily_anomaly.index.day == 1)
][
    ["sales", "rolling_median", "robust_z"]
]

jan1_check

,sales,rolling_median,robust_z
date,,,
2013-01-01,2511.618999,3.452912e+05,-4.073302
2014-01-01,8602.065404,5.634506e+05,-5.220684
2015-01-01,12773.616980,7.163296e+05,-2.413242
2016-01-01,16433.394000,9.467252e+05,-4.240582
2017-01-01,12082.500997,1.028243e+06,-4.171557


In [ ]:
fig = px.line(
    daily_anomaly.reset_index(),
    x="date",
    y="sales",
    title="Ventas diarias y anomalías temporales"
)

fig.add_scatter(
    x=anomalies.index,
    y=anomalies["sales"],
    mode="markers",
    name="Anomalías"
)

fig.show()

### Terremoto de Manabí — abril de 2016

El dataset de eventos registra el terremoto de Manabí en abril de 2016.

Se analiza este período como un posible quiebre estructural de corto plazo, comparando las ventas durante el evento y los días posteriores contra el promedio de las cuatro semanas anteriores.

In [ ]:
earthquake_events = holidays[
    holidays["description"]
    .str.contains(
        "Terremoto",
        case=False,
        na=False
    )
][
    ["date", "type", "description"]
]

earthquake_events

,date,type,description
219,2016-04-16,Event,Terremoto Manabi
220,2016-04-17,Event,Terremoto Manabi+1
221,2016-04-18,Event,Terremoto Manabi+2
222,2016-04-19,Event,Terremoto Manabi+3
223,2016-04-20,Event,Terremoto Manabi+4
225,2016-04-21,Event,Terremoto Manabi+5
226,2016-04-22,Event,Terremoto Manabi+6
227,2016-04-23,Event,Terremoto Manabi+7
228,2016-04-24,Event,Terremoto Manabi+8
229,2016-04-25,Event,Terremoto Manabi+9


In [ ]:
earthquake_start = pd.Timestamp("2016-04-16")

baseline_start = earthquake_start - pd.Timedelta(days=28)
baseline_end = earthquake_start - pd.Timedelta(days=1)

baseline_sales = daily_anomaly.loc[
    baseline_start:baseline_end,
    "sales"
].mean()

print(
    f"Promedio diario 4 semanas previas: "
    f"{baseline_sales:,.0f}"
)

Promedio diario 4 semanas previas: 761,229


In [ ]:
earthquake_period = (
    daily_anomaly
    .loc["2016-04-16":"2016-04-30", ["sales"]]
    .copy()
)

earthquake_period["variacion_vs_previo"] = (
    earthquake_period["sales"]
    / baseline_sales
    - 1
) * 100

earthquake_period

,sales,variacion_vs_previo
date,,
2016-04-16,8.621215e+05,13.253862
2016-04-17,1.271834e+06,67.076316
2016-04-18,1.345921e+06,76.808847
2016-04-19,1.152089e+06,51.345899
2016-04-20,1.062426e+06,39.567198
2016-04-21,1.001080e+06,31.508323
2016-04-22,8.570592e+05,12.588853
2016-04-23,1.022143e+06,34.275342
2016-04-24,1.039370e+06,36.538365


In [ ]:
earthquake_period.loc[
    "2016-04-16":"2016-04-18"
]

,sales,variacion_vs_previo
date,,
2016-04-16,8.621215e+05,13.253862
2016-04-17,1.271834e+06,67.076316
2016-04-18,1.345921e+06,76.808847


In [ ]:
earthquake_plot = (
    daily_anomaly
    .loc["2016-03-19":"2016-05-01"]
    .reset_index()
)

fig = px.line(
    earthquake_plot,
    x="date",
    y="sales",
    title="Impacto del terremoto de Manabí sobre las ventas"
)

fig.add_hline(
    y=baseline_sales,
    line_dash="dash",
    annotation_text="Promedio 4 semanas previas"
)

fig.add_vline(
    x=pd.Timestamp("2016-04-16").timestamp() * 1000,
    line_dash="dot",
    annotation_text="Terremoto"
)

fig.show()

### Hallazgo — anomalías temporales y terremoto de Manabí

El método global basado en IQR fue reemplazado por una detección robusta mediante **mediana móvil de 28 días y MAD**, permitiendo comparar cada observación con su contexto temporal reciente.

Con un umbral de `|robust_z| > 4` se identificaron **21 días anómalos**, frente a los 5 detectados originalmente mediante IQR. Esto confirma que un umbral global resulta inadecuado para una serie con tendencia y estacionalidad.

Cuatro de los cinco días 1 de enero superan además el umbral estadístico de anomalía. El **1 de enero de 2015** no supera dicho umbral, pero igualmente se interpreta como un día de cierre o funcionamiento excepcional a partir del conocimiento del calendario.

También se identificó un quiebre relevante asociado al **terremoto de Manabí del 16 de abril de 2016**. Frente a un promedio de **761.229 ventas diarias durante las cuatro semanas anteriores**, las ventas aumentaron aproximadamente **13,3 % el 16 de abril, 67,1 % el 17 y 76,8 % el 18**, manteniéndose elevadas durante varios días posteriores.

Este evento constituye un ejemplo de shock externo capaz de modificar temporalmente el comportamiento habitual de la demanda y justifica incorporar eventos extraordinarios dentro del calendario analítico.

## 6.11 Calendario completo de feriados y eventos

Los feriados y eventos no afectan necesariamente a todas las tiendas por igual.

Se construye un calendario con una fila por fecha y tienda, aplicando los eventos nacionales a todos los establecimientos, los regionales según el estado y los locales según la ciudad.

También se conservan los distintos tipos de evento y las fechas donde coinciden múltiples acontecimientos.

In [ ]:
from src.features.build_calendar import build_calendar

In [ ]:
calendar = build_calendar(
    sales,
    stores,
    holidays
)

calendar.head()

,date,store_nbr,city,state,type,cluster,is_additional,is_bridge,is_event,is_holiday,is_transfer,is_work_day,event_count,event_descriptions,is_new_year_closure
0,2013-01-01,1,Quito,Pichincha,D,13,0,0,0,1,0,0,1,Primer dia del ano,1
1,2013-01-01,2,Quito,Pichincha,D,13,0,0,0,1,0,0,1,Primer dia del ano,1
2,2013-01-01,3,Quito,Pichincha,D,8,0,0,0,1,0,0,1,Primer dia del ano,1
3,2013-01-01,4,Quito,Pichincha,D,9,0,0,0,1,0,0,1,Primer dia del ano,1
4,2013-01-01,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4,0,0,0,1,0,0,1,Primer dia del ano,1


In [ ]:
print(f"Filas calendario: {len(calendar):,}")
print(f"Fechas: {calendar['date'].nunique():,}")
print(f"Tiendas: {calendar['store_nbr'].nunique()}")

Filas calendario: 91,152
Fechas: 1,688
Tiendas: 54


In [ ]:
calendar[
    [
        "is_holiday",
        "is_event",
        "is_transfer",
        "is_additional",
        "is_bridge",
        "is_work_day"
    ]
].sum()

is_holiday       2651
is_event         2970
is_transfer       389
is_additional    1678
is_bridge         162
is_work_day       270
dtype: int64

In [ ]:
multiple_events = calendar[
    calendar["event_count"] > 1
]

print(
    "Combinaciones fecha-tienda con múltiples eventos:",
    len(multiple_events)
)

multiple_events[
    [
        "date",
        "store_nbr",
        "city",
        "state",
        "event_count",
        "event_descriptions"
    ]
].head(20)

Combinaciones fecha-tienda con múltiples eventos: 239


,date,store_nbr,city,state,event_count,event_descriptions
7095,2013-05-12,22,Puyo,Pastaza,2,Cantonizacion del Puyo | Dia de la Madre
19194,2013-12-22,25,Salinas,Santa Elena,2,Cantonizacion de Salinas | Navidad-3
29171,2014-06-25,12,Latacunga,Cotopaxi,2,Cantonizacion de Latacunga | Mundial de futbol...
29172,2014-06-25,13,Latacunga,Cotopaxi,2,Cantonizacion de Latacunga | Mundial de futbol...
29174,2014-06-25,15,Ibarra,Imbabura,2,Mundial de futbol Brasil: Ecuador-Francia | Pr...
29199,2014-06-25,40,Machala,El Oro,2,Fundacion de Machala | Mundial de futbol Brasi...
29200,2014-06-25,41,Machala,El Oro,2,Fundacion de Machala | Mundial de futbol Brasi...
38904,2014-12-22,25,Salinas,Santa Elena,2,Cantonizacion de Salinas | Navidad-3
39096,2014-12-26,1,Quito,Pichincha,2,Navidad+1 | Puente Navidad
39097,2014-12-26,2,Quito,Pichincha,2,Navidad+1 | Puente Navidad


In [ ]:
calendar[
    calendar["date"].isin(missing_dates)
][
    [
        "date",
        "store_nbr",
        "city",
        "is_holiday",
        "is_event",
        "is_work_day"
    ]
].head(20)

,date,store_nbr,city,is_holiday,is_event,is_work_day
19332,2013-12-25,1,Quito,1,0,0
19333,2013-12-25,2,Quito,1,0,0
19334,2013-12-25,3,Quito,1,0,0
19335,2013-12-25,4,Quito,1,0,0
19336,2013-12-25,5,Santo Domingo,1,0,0
19337,2013-12-25,6,Quito,1,0,0
19338,2013-12-25,7,Quito,1,0,0
19339,2013-12-25,8,Quito,1,0,0
19340,2013-12-25,9,Quito,1,0,0
19341,2013-12-25,10,Quito,1,0,0


In [ ]:
print(
    "Duplicados fecha-tienda:",
    calendar.duplicated(["date", "store_nbr"]).sum()
)

print(
    "Registros de cierre por Año Nuevo:",
    calendar["is_new_year_closure"].sum()
)

Duplicados fecha-tienda: 0
Registros de cierre por Año Nuevo: 270


### Hallazgo — calendario comercial por tienda

El calendario original de feriados no puede tratarse únicamente como una variable binaria por fecha, ya que existen eventos de diferente alcance territorial y múltiples acontecimientos pueden coincidir en un mismo día.

Se construyó un calendario completo de **91.152 combinaciones fecha–tienda**, correspondientes a **1.688 días y 54 establecimientos**, aplicando los eventos nacionales a todas las tiendas, los regionales según el estado y los locales según la ciudad.

Además, se conservaron por separado los tipos `Holiday`, `Event`, `Transfer`, `Additional`, `Bridge` y `Work Day`. Se identificaron **239 combinaciones fecha–tienda afectadas por más de un evento**, evitando la pérdida de información que producía el uso previo de `drop_duplicates("date")`.

El calendario también permite representar las cuatro fechas ausentes del panel de ventas —los 25 de diciembre de 2013 a 2016— y distinguir explícitamente los cierres asociados al 1 de enero.

Esta estructura deja preparado un insumo temporal a nivel tienda que podrá integrarse posteriormente al modelo sin asumir que todos los eventos afectan de igual forma a todos los establecimientos.

## 6.12 Estructura de las series tienda–familia

El problema de demanda está compuesto por múltiples series temporales independientes correspondientes a combinaciones tienda–familia.

Se analiza el comportamiento de cada serie durante 2017 para identificar series activas, intermitentes y sin ventas, además de estudiar la concentración del volumen comercial entre las distintas combinaciones.

In [ ]:
sales_2017 = sales_valid[
    sales_valid["date"].dt.year == 2017
].copy()

series_2017 = (
    sales_2017
    .groupby(
        ["store_nbr", "family"],
        as_index=False
    )
    .agg(
        registros=("sales", "size"),
        dias_cero=("sales", lambda x: (x == 0).sum()),
        ventas_totales=("sales", "sum"),
        ventas_promedio=("sales", "mean")
    )
)

series_2017["pct_ceros"] = (
    series_2017["dias_cero"]
    / series_2017["registros"]
    * 100
)

print(
    "Cantidad de series tienda-familia:",
    len(series_2017)
)

series_2017.head()

Cantidad de series tienda-familia: 1782


,store_nbr,family,registros,dias_cero,ventas_totales,ventas_promedio,pct_ceros
0,1,AUTOMOTIVE,227,10,924.0,4.070485,4.405286
1,1,BABY CARE,227,227,0.0,0.000000,100.000000
2,1,BEAUTY,227,17,736.0,3.242291,7.488987
3,1,BEVERAGES,227,1,493224.0,2172.792952,0.440529
4,1,BOOKS,227,151,104.0,0.458150,66.519824


In [ ]:
series_muertas = (
    series_2017["pct_ceros"] == 100
).sum()

series_muy_intermitentes = (
    series_2017["pct_ceros"] > 80
).sum()

series_intermitentes_no_muertas = (
    (series_2017["pct_ceros"] > 80) &
    (series_2017["pct_ceros"] < 100)
).sum()

print(f"Series 100% en cero: {series_muertas}")
print(
    f"Series con más de 80% de ceros: "
    f"{series_muy_intermitentes}"
)
print(
    f"Intermitentes >80% excluyendo muertas: "
    f"{series_intermitentes_no_muertas}"
)

Series 100% en cero: 67
Series con más de 80% de ceros: 132
Intermitentes >80% excluyendo muertas: 65


In [ ]:
series_2017["tipo_serie"] = np.select(
    [
        series_2017["pct_ceros"] == 100,
        series_2017["pct_ceros"] > 80
    ],
    [
        "Muerta",
        "Intermitente"
    ],
    default="Activa"
)

series_type_summary = (
    series_2017["tipo_serie"]
    .value_counts()
    .rename_axis("tipo_serie")
    .reset_index(name="cantidad")
)

series_type_summary

,tipo_serie,cantidad
0,Activa,1650
1,Muerta,67
2,Intermitente,65


In [ ]:
fig = px.bar(
    series_type_summary,
    x="tipo_serie",
    y="cantidad",
    text="cantidad",
    title="Perfil de series tienda–familia en 2017"
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Tipo de serie",
    yaxis_title="Cantidad de series"
)

fig.show()

In [ ]:
series_rank = (
    series_2017
    .sort_values(
        "ventas_totales",
        ascending=False
    )
    .reset_index(drop=True)
)

series_rank["ranking"] = (
    series_rank.index + 1
)

total_sales_2017 = (
    series_rank["ventas_totales"].sum()
)

series_rank["participacion_acumulada"] = (
    series_rank["ventas_totales"]
    .cumsum()
    / total_sales_2017
    * 100
)

top_200_share = (
    series_rank
    .head(200)["ventas_totales"]
    .sum()
    / total_sales_2017
    * 100
)

print(
    f"Participación de las 200 series principales: "
    f"{top_200_share:.2f}%"
)

Participación de las 200 series principales: 76.50%


In [ ]:
fig = px.line(
    series_rank,
    x="ranking",
    y="participacion_acumulada",
    title="Concentración acumulada de ventas por serie"
)

fig.add_vline(
    x=200,
    line_dash="dot"
)

fig.add_hline(
    y=top_200_share,
    line_dash="dash"
)

fig.update_layout(
    xaxis_title="Ranking de series",
    yaxis_title="% acumulado de ventas"
)

fig.show()

In [ ]:
store_sales_2017 = (
    sales_2017
    .groupby(
        "store_nbr",
        as_index=False
    )
    .agg(
        ventas_totales=("sales", "sum")
    )
)

store_sales_2017 = store_sales_2017.merge(
    stores[
        [
            "store_nbr",
            "city",
            "state",
            "type",
            "cluster"
        ]
    ],
    on="store_nbr",
    how="left"
)

In [ ]:
cluster_profile = (
    store_sales_2017
    .groupby(
        "cluster",
        as_index=False
    )
    .agg(
        tiendas=("store_nbr", "nunique"),
        ventas_totales=("ventas_totales", "sum"),
        ventas_promedio_tienda=("ventas_totales", "mean")
    )
    .sort_values(
        "ventas_promedio_tienda",
        ascending=False
    )
)

cluster_profile

,cluster,tiendas,ventas_totales,ventas_promedio_tienda
4,5,1,1.049012e+07,1.049012e+07
10,11,3,2.073634e+07,6.912113e+06
13,14,4,2.723782e+07,6.809454e+06
7,8,3,1.874111e+07,6.247036e+06
16,17,1,5.364285e+06,5.364285e+06
5,6,6,2.190492e+07,3.650820e+06
11,12,1,3.367259e+06,3.367259e+06
12,13,4,1.295512e+07,3.238780e+06
0,1,3,9.702770e+06,3.234257e+06
1,2,2,6.251281e+06,3.125640e+06


In [ ]:
fig = px.bar(
    cluster_profile,
    x="cluster",
    y="ventas_promedio_tienda",
    text="tiendas",
    title="Ventas promedio por tienda según cluster"
)

fig.update_layout(
    xaxis_title="Cluster",
    yaxis_title="Ventas promedio por tienda"
)

fig.show()

In [ ]:
city_profile = (
    store_sales_2017
    .groupby(
        "city",
        as_index=False
    )
    .agg(
        tiendas=("store_nbr", "nunique"),
        ventas_totales=("ventas_totales", "sum")
    )
    .sort_values(
        "ventas_totales",
        ascending=False
    )
)

print(
    f"Ciudades representadas: "
    f"{store_sales_2017['city'].nunique()}"
)

city_profile.head(10)

Ciudades representadas: 22


,city,tiendas,ventas_totales
18,Quito,18,9.835858e+07
8,Guayaquil,8,2.182327e+07
3,Cuenca,3,9.990351e+06
0,Ambato,2,6.821124e+06
21,Santo Domingo,3,6.751434e+06
13,Machala,2,6.450059e+06
14,Manta,2,5.407709e+06
2,Cayambe,1,4.624729e+06
4,Daule,1,3.507029e+06
10,Latacunga,2,3.369019e+06


### Hallazgo — heterogeneidad de las series tienda–familia

El problema de demanda está compuesto por **1.782 series temporales tienda–familia**, pero su comportamiento no es homogéneo.

Durante 2017 se identificaron **67 series con ventas iguales a cero en el 100 % de sus registros** y **132 series con más del 80 % de observaciones sin ventas**. Excluyendo las series completamente inactivas, quedan **65 series altamente intermitentes**, mientras que **1.650 series presentan un comportamiento predominantemente activo**.

Además, el volumen comercial se encuentra fuertemente concentrado: las **200 series con mayores ventas explican aproximadamente el 76,50 % de las ventas totales de 2017**.

Esto indica que no todas las combinaciones tienda–familia deberían tratarse de la misma manera en etapas posteriores. Las series activas concentran la mayor parte del negocio, mientras que las series intermitentes o sin ventas requerirán un tratamiento diferenciado.

El análisis de `stores.csv` también muestra diferencias relevantes entre los **17 clusters de tiendas** y entre las **22 ciudades representadas**, por lo que variables como `cluster`, `city`, `state` y `type` constituyen información potencialmente útil para caracterizar la demanda.

Estos resultados justifican una estrategia de modelado que contemple tanto la heterogeneidad entre series como la concentración del volumen comercial.

## 6.13 Estrategia de validación temporal y baseline

Al tratarse de un problema de pronóstico temporal, los datos no pueden dividirse aleatoriamente entre entrenamiento y validación.

Se reservan los **últimos 15 días disponibles** como conjunto de validación, simulando una situación real donde el modelo debe predecir un período futuro utilizando únicamente información conocida previamente.

Como referencia inicial se construye un **baseline estacional semanal**, basado exclusivamente en las cuatro semanas anteriores al inicio de la validación.

La métrica seleccionada es **RMSLE (Root Mean Squared Logarithmic Error)**, adecuada para comparar errores relativos en una variable de demanda altamente asimétrica.

In [ ]:
validation_end = sales_valid["date"].max()
validation_start = validation_end - pd.Timedelta(days=14)

train_temporal = sales_valid[
    sales_valid["date"] < validation_start
].copy()

validation_temporal = sales_valid[
    sales_valid["date"] >= validation_start
].copy()

print("Entrenamiento:")
print(
    train_temporal["date"].min().date(),
    "→",
    train_temporal["date"].max().date()
)

print("\nValidación:")
print(
    validation_temporal["date"].min().date(),
    "→",
    validation_temporal["date"].max().date()
)

print(f"\nDías de validación: {validation_temporal['date'].nunique()}")
print(f"Registros de validación: {len(validation_temporal):,}")

Entrenamiento:
2013-01-01 → 2017-07-31

Validación:
2017-08-01 → 2017-08-15

Días de validación: 15
Registros de validación: 26,730


In [ ]:
baseline_start = validation_start - pd.Timedelta(days=28)

baseline_source = train_temporal[
    train_temporal["date"] >= baseline_start
].copy()

baseline_source["day_of_week"] = (
    baseline_source["date"].dt.dayofweek
)

print(
    "Período utilizado para construir baseline:",
    baseline_source["date"].min().date(),
    "→",
    baseline_source["date"].max().date()
)

Período utilizado para construir baseline: 2017-07-04 → 2017-07-31


In [ ]:
seasonal_baseline = (
    baseline_source
    .groupby(
        ["store_nbr", "family", "day_of_week"],
        as_index=False
    )
    .agg(
        pred_baseline=("sales", "mean")
    )
)

seasonal_baseline.head()

,store_nbr,family,day_of_week,pred_baseline
0,1,AUTOMOTIVE,0,4.25
1,1,AUTOMOTIVE,1,6.25
2,1,AUTOMOTIVE,2,4.25
3,1,AUTOMOTIVE,3,5.75
4,1,AUTOMOTIVE,4,6.25


In [ ]:
validation_eval = validation_temporal.copy()

validation_eval["day_of_week"] = (
    validation_eval["date"].dt.dayofweek
)

validation_eval = validation_eval.merge(
    seasonal_baseline,
    on=["store_nbr", "family", "day_of_week"],
    how="left"
)

In [ ]:
fallback = (
    train_temporal
    .groupby(
        ["store_nbr", "family"],
        as_index=False
    )
    .agg(
        fallback_mean=("sales", "mean")
    )
)

validation_eval = validation_eval.merge(
    fallback,
    on=["store_nbr", "family"],
    how="left"
)

validation_eval["pred_baseline"] = (
    validation_eval["pred_baseline"]
    .fillna(validation_eval["fallback_mean"])
    .fillna(0)
    .clip(lower=0)
)

In [ ]:
rmsle_baseline = np.sqrt(
    np.mean(
        (
            np.log1p(validation_eval["sales"])
            -
            np.log1p(validation_eval["pred_baseline"])
        ) ** 2
    )
)

print(
    f"RMSLE baseline estacional: "
    f"{rmsle_baseline:.4f}"
)

RMSLE baseline estacional: 0.5285


In [ ]:
baseline_daily = (
    validation_eval
    .groupby("date", as_index=False)
    .agg(
        ventas_reales=("sales", "sum"),
        baseline=("pred_baseline", "sum")
    )
)

baseline_plot = baseline_daily.melt(
    id_vars="date",
    value_vars=["ventas_reales", "baseline"],
    var_name="serie",
    value_name="ventas"
)

fig = px.line(
    baseline_plot,
    x="date",
    y="ventas",
    color="serie",
    markers=True,
    title="Validación temporal — ventas reales vs baseline estacional"
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Ventas"
)

fig.show()

### Hallazgo — estrategia de validación y baseline

Debido a la naturaleza temporal del problema, se descartó una partición aleatoria de los datos. Se reservaron los **últimos 15 días disponibles**, correspondientes al período **1 al 15 de agosto de 2017**, como conjunto de validación, manteniendo todo el historial anterior como información de entrenamiento.

Como referencia inicial se construyó un **baseline estacional semanal** utilizando exclusivamente las cuatro semanas previas al inicio de la validación y diferenciando cada combinación tienda–familia según el día de la semana.

El baseline obtuvo un **RMSLE de 0,5285**. Este resultado establece un punto de referencia cuantitativo que deberá ser superado por los modelos predictivos posteriores para justificar su mayor complejidad.

A nivel agregado, el baseline reproduce adecuadamente buena parte de la dinámica semanal de las ventas, aunque presenta desviaciones relevantes en determinados días, evidenciando que la estacionalidad semanal por sí sola no explica completamente la demanda.

La estrategia definida permite evaluar futuros modelos bajo una simulación más realista: predecir información futura utilizando exclusivamente datos disponibles hasta el momento de realizar el pronóstico.

# Síntesis del EDA preliminar

El análisis exploratorio muestra que la demanda de Corporación Favorita presenta una estructura fuertemente temporal, heterogénea y concentrada entre tiendas y familias de productos.

Los registros con `sales = 0` no representan un único fenómeno. Del 31,30 % de ceros presentes en el panel original, una parte corresponde a períodos previos a la apertura de tiendas o a familias que determinados establecimientos no comercializan. Una vez considerados estos casos estructurales, los ceros restantes representan períodos reales sin ventas y constituyen información relevante sobre la dinámica de la demanda.

La demanda presenta una marcada estacionalidad. Los fines de semana, especialmente los domingos, registran niveles considerablemente superiores al promedio, mientras que los jueves muestran sistemáticamente menores ventas. También se observan patrones dentro del mes, con mayor actividad al comienzo, alrededor de la quincena y hacia el cierre mensual. Estos resultados justifican incorporar variables de calendario y rezagos temporales.

Las promociones presentan una asociación positiva con las ventas incluso al comparar observaciones dentro de una misma combinación tienda–familia desde 2014. El efecto, sin embargo, varía considerablemente entre familias y debe interpretarse como asociación y no como evidencia causal.

Los feriados y eventos tampoco presentan un efecto uniforme. Su impacto depende del tipo de acontecimiento y de su alcance nacional, regional o local. El calendario construido a nivel fecha–tienda permite conservar esta información y representar correctamente fechas con múltiples eventos.

El análisis robusto de anomalías mediante mediana móvil y MAD identificó 21 días con comportamiento excepcional. Estos valores no serán eliminados automáticamente, ya que varios corresponden a acontecimientos reales. El terremoto de Manabí de abril de 2016 constituye un ejemplo claro: las ventas aumentaron fuertemente durante los días posteriores al evento. Asimismo, los días 1 de enero muestran un comportamiento excepcional asociado al funcionamiento comercial y no deben tratarse como demanda ordinaria.

El problema está compuesto por 1.782 series tienda–familia. En 2017, 1.650 presentan actividad regular, 65 muestran demanda altamente intermitente y 67 permanecen sin ventas. Además, las 200 series de mayor volumen concentran aproximadamente el 76,5 % de las ventas, evidenciando una fuerte concentración comercial.

Las transacciones presentan una elevada relación contemporánea con las ventas (r = 0,837), que también se mantiene al analizar cambios diarios (r = 0,783). Sin embargo, las transacciones del mismo día no pueden utilizarse directamente en un pronóstico ex ante. El rezago de siete días conserva una asociación importante (r = 0,605), consistente con la estacionalidad semanal observada.

El precio del petróleo presenta una correlación negativa moderada con las ventas cuando se comparan sus niveles (r = -0,627). No obstante, esta asociación prácticamente desaparece al analizar primeras diferencias y rezagos, por lo que su capacidad explicativa de corto plazo parece limitada bajo las transformaciones estudiadas.

Finalmente, se definió una estrategia de validación estrictamente temporal, reservando los últimos 15 días disponibles como conjunto de validación. Como referencia inicial se construyó un baseline estacional semanal utilizando exclusivamente información previa, obteniendo un RMSLE de 0,5285. Este resultado servirá como referencia mínima para evaluar posteriormente los modelos predictivos.
